# Main Analysis

Phase 3 establishes reproducible data loading and initial dataset inspection. It does not include exploratory plots, feature transformation, or model training.

## 1. Project setup

The setup locates the repository without embedding a machine-specific path and imports the shared loading utilities.

In [1]:
from pathlib import Path
import platform
import sys

import pandas as pd
from IPython.display import Markdown, display

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
if not (project_root / 'src').is_dir():
    raise ValueError('Run this notebook from the project root or the notebooks directory.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loading import (
    detect_constant_columns,
    detect_duplicate_columns,
    get_data_paths,
    load_available_transaction_files,
    load_transactions,
    summarize_dataframe,
    validate_transaction_schema,
)

pd.set_option('display.max_columns', 20)
print(f'Python {platform.python_version()} | pandas {pd.__version__}')

Python 3.12.13 | pandas 2.2.3


## 2. Dataset source and scope

The data come from the [Fraud Detection Handbook simulated-data repository](https://github.com/Fraud-Detection-Handbook/simulated-data-raw). Each file contains one day of labeled payment-card transactions. This notebook inspects the raw fields only; the handbook's transformed dataset is outside this phase.

The data are synthetic. Their transparency supports internal reproduction, but results on them cannot by themselves establish performance on a real payment system.

## 3. Data loading

The loader reads daily pickle files in date order and performs no preprocessing. If the data are missing, it stops with the official source and the exact acquisition command.

In [2]:
data_paths = get_data_paths(project_root)
transaction_files = load_available_transaction_files(data_paths['transactions'])
print(
    f'Discovered {len(transaction_files)} daily files: '
    f'{transaction_files[0].name} through {transaction_files[-1].name}.'
)

transactions = load_transactions(data_paths['transactions'])
print('Transaction files loaded without feature transformation.')

Discovered 183 daily files: 2018-04-01.pkl through 2018-09-30.pkl.


Transaction files loaded without feature transformation.


## 4. Schema validation

Schema validation checks required fields, datetime representation, and the documented fraud-label domains. It does not silently coerce invalid data.

In [3]:
validate_transaction_schema(transactions)
print('Schema validation passed for the documented raw transaction fields.')

Schema validation passed for the documented raw transaction fields.


`TX_FRAUD` is the binary prediction target. `TX_FRAUD_SCENARIO` explains which simulator rule assigned a fraud label and must never be used as a predictor because it directly reveals the target-generation mechanism.

## 5. Dataset size

The structural summary records the observation count, field count, in-memory size, missing cells, duplicate rows, and index representation.

In [4]:
dataset_summary = summarize_dataframe(transactions)
display(dataset_summary.to_frame())
display(
    Markdown(
        f"The loaded table contains **{int(dataset_summary['rows']):,} transactions** "
        f"and **{int(dataset_summary['columns'])} raw columns**. Each row is one "
        "labeled transaction event rather than an aggregated customer record."
    )
)

,value
rows,1754155
columns,9
memory_mib,307.81
missing_cells,0
duplicate_rows,0
index_type,RangeIndex
index_name,<unnamed>


The loaded table contains **1,754,155 transactions** and **9 raw columns**. Each row is one labeled transaction event rather than an aggregated customer record.

## 6. Column and index inspection

Column names should preserve the simulator's documented semantics. The DataFrame index is inspected separately from the transaction identifier.

In [5]:
column_positions = pd.DataFrame(
    {'position': range(len(transactions.columns)), 'column': transactions.columns}
)
display(column_positions)

index_inspection = pd.Series(
    {
        'index_type': type(transactions.index).__name__,
        'index_name': transactions.index.name or '<unnamed>',
        'index_is_unique': transactions.index.is_unique,
        'transaction_id_is_unique': transactions['TRANSACTION_ID'].is_unique,
    },
    name='value',
)
display(index_inspection.to_frame())

,position,column
0,0,TRANSACTION_ID
1,1,TX_DATETIME
2,2,CUSTOMER_ID
3,3,TERMINAL_ID
4,4,TX_AMOUNT
5,5,TX_TIME_SECONDS
6,6,TX_TIME_DAYS
7,7,TX_FRAUD
8,8,TX_FRAUD_SCENARIO


,value
index_type,RangeIndex
index_name,<unnamed>
index_is_unique,True
transaction_id_is_unique,True


In [6]:
id_interpretation = (
    '`TRANSACTION_ID` is unique and is the meaningful event identifier.'
    if transactions['TRANSACTION_ID'].is_unique
    else '`TRANSACTION_ID` is not unique; this requires investigation before modeling.'
)
display(
    Markdown(
        'The unnamed `RangeIndex` is only an in-memory row position after concatenation. '
        + id_interpretation
        + ' Identifier columns should not be treated as continuous predictors.'
    )
)

The unnamed `RangeIndex` is only an in-memory row position after concatenation. `TRANSACTION_ID` is unique and is the meaningful event identifier. Identifier columns should not be treated as continuous predictors.

## 7. Feature types

Storage dtypes are paired with analytical roles so that identifiers, timestamps, predictors, and labels are not confused.

In [7]:
semantic_roles = {
    'TRANSACTION_ID': 'event identifier; exclude from predictors',
    'TX_DATETIME': 'event time; use for ordering and temporal splitting',
    'CUSTOMER_ID': 'entity identifier; categorical semantics',
    'TERMINAL_ID': 'entity identifier; categorical semantics',
    'TX_AMOUNT': 'continuous transaction attribute',
    'TX_TIME_SECONDS': 'elapsed-time representation',
    'TX_TIME_DAYS': 'day index for temporal grouping',
    'TX_FRAUD': 'binary target; exclude from predictors',
    'TX_FRAUD_SCENARIO': 'simulator explanation label; leakage if used as predictor',
}

feature_types = pd.DataFrame(
    {
        'dtype': transactions.dtypes.astype(str),
        'non_missing': transactions.notna().sum(),
        'distinct_values': transactions.nunique(dropna=False),
        'analytical_role': pd.Series(semantic_roles),
    }
)
display(feature_types)

,dtype,non_missing,distinct_values,analytical_role
TRANSACTION_ID,int64,1754155,1754155,event identifier; exclude from predictors
TX_DATETIME,datetime64[ns],1754155,1635076,event time; use for ordering and temporal spli...
CUSTOMER_ID,object,1754155,4990,entity identifier; categorical semantics
TERMINAL_ID,object,1754155,10000,entity identifier; categorical semantics
TX_AMOUNT,float64,1754155,24585,continuous transaction attribute
TX_TIME_SECONDS,object,1754155,1635076,elapsed-time representation
TX_TIME_DAYS,object,1754155,183,day index for temporal grouping
TX_FRAUD,int64,1754155,2,binary target; exclude from predictors
TX_FRAUD_SCENARIO,int64,1754155,4,simulator explanation label; leakage if used a...


Customer and terminal identifiers are stored as integers, but their numbers do not represent magnitude. Treating them as ordinary continuous quantities would create meaningless distance relationships.

## 8. Missing-value analysis

Missingness is measured per field before any imputation decision.

In [8]:
missing_values = pd.DataFrame(
    {
        'missing_count': transactions.isna().sum(),
        'missing_percent': transactions.isna().mean().mul(100).round(4),
    }
)
display(missing_values)

if missing_values['missing_count'].sum() == 0:
    missing_interpretation = (
        'No missing cells were found. This is consistent with a controlled simulator, '
        'but it understates the incomplete records expected in operational payment data.'
    )
else:
    missing_interpretation = (
        'Missing values are present and must be explained before any imputation; '
        'blind filling could erase fraud-relevant collection failures.'
    )
display(Markdown(missing_interpretation))

,missing_count,missing_percent
TRANSACTION_ID,0,0.0
TX_DATETIME,0,0.0
CUSTOMER_ID,0,0.0
TERMINAL_ID,0,0.0
TX_AMOUNT,0,0.0
TX_TIME_SECONDS,0,0.0
TX_TIME_DAYS,0,0.0
TX_FRAUD,0,0.0
TX_FRAUD_SCENARIO,0,0.0


No missing cells were found. This is consistent with a controlled simulator, but it understates the incomplete records expected in operational payment data.

## 9. Duplicate-row analysis

Exact duplicate events may indicate accidental repeated ingestion. Repeated customers, terminals, amounts, or timestamps alone are not duplicate transactions.

In [9]:
duplicate_row_count = int(dataset_summary['duplicate_rows'])
duplicate_row_percent = duplicate_row_count / len(transactions) * 100
display(
    pd.Series(
        {
            'duplicate_rows': duplicate_row_count,
            'duplicate_percent': round(duplicate_row_percent, 6),
        },
        name='value',
    ).to_frame()
)

duplicate_interpretation = (
    'No exact duplicate transaction rows were found.'
    if duplicate_row_count == 0
    else 'Exact duplicate rows exist and may inflate both prevalence and validation scores.'
)
display(Markdown(duplicate_interpretation))

,value
duplicate_rows,0.0
duplicate_percent,0.0


No exact duplicate transaction rows were found.

## 10. Constant-feature and duplicate-feature analysis

A constant column has no discriminatory information. An exact duplicate column adds no new information and can distort interpretation or unnecessarily increase computation.

In [10]:
constant_columns = detect_constant_columns(transactions)
duplicate_columns = detect_duplicate_columns(transactions)
display(
    pd.Series(
        {
            'constant_columns': constant_columns or '<none>',
            'duplicate_columns': duplicate_columns or '<none>',
        },
        name='finding',
    ).to_frame()
)

if not constant_columns and not duplicate_columns:
    feature_redundancy_interpretation = (
        'No constant or exactly duplicated raw columns were found. This does not rule '
        'out high correlation or semantic redundancy, which belongs to later analysis.'
    )
else:
    feature_redundancy_interpretation = (
        'Constant or duplicate fields should be excluded from predictive inputs after '
        'their provenance is checked.'
    )
display(Markdown(feature_redundancy_interpretation))

,finding
constant_columns,<none>
duplicate_columns,<none>


No constant or exactly duplicated raw columns were found. This does not rule out high correlation or semantic redundancy, which belongs to later analysis.

## 11. Target/class prevalence

Fraud prevalence determines whether accuracy is informative and establishes the operational scale of false positives and false negatives.

In [11]:
class_counts = transactions['TX_FRAUD'].value_counts().sort_index()
class_prevalence = pd.DataFrame(
    {
        'label_name': ['legitimate', 'fraud'],
        'count': class_counts.reindex([0, 1], fill_value=0),
    },
    index=pd.Index([0, 1], name='TX_FRAUD'),
)
class_prevalence['percent'] = (
    class_prevalence['count'] / class_prevalence['count'].sum() * 100
).round(4)
display(class_prevalence)

fraud_count = int(class_prevalence.loc[1, 'count'])
legitimate_count = int(class_prevalence.loc[0, 'count'])
fraud_rate = fraud_count / len(transactions)
imbalance_ratio = legitimate_count / fraud_count if fraud_count else float('inf')
display(
    Markdown(
        f'Fraud prevalence is **{fraud_rate:.3%}**, or approximately one fraud for '
        f'every **{imbalance_ratio:.1f} legitimate transactions**. A classifier that '
        'predicts every event as legitimate would appear accurate while detecting no '
        'fraud, so accuracy alone is unsuitable.'
    )
)

,label_name,count,percent
TX_FRAUD,,,
0,legitimate,1739474,99.1631
1,fraud,14681,0.8369


Fraud prevalence is **0.837%**, or approximately one fraud for every **118.5 legitimate transactions**. A classifier that predicts every event as legitimate would appear accurate while detecting no fraud, so accuracy alone is unsuitable.

## 12. Initial temporal coverage check

Only coverage and ordering are checked here. Temporal distributions and concept drift belong to later EDA.

In [12]:
start_timestamp = transactions['TX_DATETIME'].min()
end_timestamp = transactions['TX_DATETIME'].max()
calendar_days = (end_timestamp.normalize() - start_timestamp.normalize()).days + 1
temporal_summary = pd.Series(
    {
        'first_timestamp': start_timestamp,
        'last_timestamp': end_timestamp,
        'calendar_days_covered': calendar_days,
        'daily_files': len(transaction_files),
        'rows_are_chronological': transactions['TX_DATETIME'].is_monotonic_increasing,
    },
    name='value',
)
display(temporal_summary.to_frame())
display(
    Markdown(
        f'The transactions cover **{calendar_days} calendar days**. Time is part of the '
        'security context: randomly mixing later transactions into training could expose '
        'the model to patterns unavailable at the historical prediction date. Later '
        'validation must therefore preserve chronology and the handbook feedback delay.'
    )
)

,value
first_timestamp,2018-04-01 00:00:31
last_timestamp,2018-09-30 23:59:57
calendar_days_covered,183
daily_files,183
rows_are_chronological,True


The transactions cover **183 calendar days**. Time is part of the security context: randomly mixing later transactions into training could expose the model to patterns unavailable at the historical prediction date. Later validation must therefore preserve chronology and the handbook feedback delay.

## 13. Short cybersecurity interpretation

This section consolidates only the verified inspection findings.

In [13]:
integrity_statement = (
    'No missing cells, exact duplicate rows, constant columns, or duplicate columns '
    'were detected in the raw table.'
    if (
        missing_values['missing_count'].sum() == 0
        and duplicate_row_count == 0
        and not constant_columns
        and not duplicate_columns
    )
    else 'The integrity findings above require resolution before predictive modeling.'
)

display(
    Markdown(
        f'- {integrity_statement}\n'
        f'- Fraud prevalence is {fraud_rate:.3%}; class-sensitive metrics will be required.\n'
        '- `TRANSACTION_ID` is an event key, while `TX_FRAUD_SCENARIO` is an explanatory '
        'label that would leak the target-generation rule if used as a feature.\n'
        '- The dataset is temporally ordered and must be split by time rather than by '
        'random row assignment.\n'
        '- These checks support internal data integrity only. Synthetic cleanliness and '
        'known fraud rules do not establish external validity.'
    )
)

- No missing cells, exact duplicate rows, constant columns, or duplicate columns were detected in the raw table.
- Fraud prevalence is 0.837%; class-sensitive metrics will be required.
- `TRANSACTION_ID` is an event key, while `TX_FRAUD_SCENARIO` is an explanatory label that would leak the target-generation rule if used as a feature.
- The dataset is temporally ordered and must be split by time rather than by random row assignment.
- These checks support internal data integrity only. Synthetic cleanliness and known fraud rules do not establish external validity.